In [104]:
import sys
import os

# Adiciona o diretório pai (raiz do projeto) ao path
sys.path.append(os.path.abspath(os.path.join('..')))

In [105]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# Tokenizador

In [106]:
from transformers import AutoTokenizer

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

text = '''Theoretical analysis and computational modeling are important tools for characterizing what 
nervous systems do, determining how they function, and understanding why they operate in particular 
ways. Neuroscience encompasses approaches ranging from molecular and cellular studies to human psychophysics 
and psychology. Theoretical neuroscience encourages cross-talk among these sub-disciplines by constructing 
compact representations of what has been learned, building bridges between different levels of description, 
and identifying unifying concepts and principles. In this book, we present the basic methods used for these 
purposes and discuss examples in which theoretical approaches have yielded insight into nervous system function.'''

In [107]:
input = tokenizer(
    text,
    padding = 'max_length',
    truncation = True,
    max_lenght = 512,
    return_tensors = 'pt'
)

In [108]:
input_ids = input['input_ids']
print(type(input_ids))
print(input_ids.shape)

<class 'torch.Tensor'>
torch.Size([1, 512])


In [109]:
vocab_size = tokenizer.vocab_size
d_model = 512

In [110]:
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=d_model)
input_embedding = embedding_layer(input_ids)

In [111]:
input_embedding.shape

torch.Size([1, 512, 512])

# Positional Enconding

In [112]:
class PosEmbeding(nn.Module):
    def __init__(self, seq, d_model):
        super().__init__()
        self.seq = seq
        self.d_model = d_model

        pe = self.forward()
        self.register_buffer('pe', pe)

# Positional Embedding Function
    def forward(self):

        pe = torch.zeros((self.seq, self.d_model))
        

        for pos in range(self.seq):
            for i in range(0, self.d_model, 2):
                pe[pos, i] = math.sin(pos / 10_000 ** (i/self.d_model))
                pe[pos, i+1] = math.cos(pos / 10_000 ** (i/self.d_model))

        return pe

# Mecanismo de Atenção

In [113]:
# Multi-Head-Attention Mecanism
class QKVAttention(nn.Module):
    def __init__(self, d_model, h, seq, causal=False):
        super().__init__()

        self.h = h
        self.reduced_dimension = d_model // h
        self.seq = seq
        self.d_model = d_model
        self.batch = 1 # Posso tornar isso dinâmico
        self.causal = causal

        if causal:
            self.register_buffer('mask', torch.tril(torch.ones(self.seq, self.seq)))
        else:
            self.mask = None

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):

        Q, K, V = self.W_Q(x), self.W_K(x), self.W_V(x)

        Q_i = Q.reshape(self.batch, self.seq, self.h, self.reduced_dimension).transpose(1, 2)
        K_i = K.reshape(self.batch, self.seq, self.h, self.reduced_dimension).transpose(1, 2)
        V_i = V.reshape(self.batch, self.seq, self.h, self.reduced_dimension).transpose(1, 2)

        score = (Q_i @ K_i.transpose(-2, -1)) / math.sqrt(self.reduced_dimension)

        if self.causal:        
            score = score.masked_fill(self.mask == 0, float('-inf'))
   
        weight = F.softmax(score, dim=-1) @ V_i
        weight = weight.transpose(1, 2).contiguous().view(self.batch, self.seq, self.d_model)


        return self.W_O(weight).squeeze(0)
    

class FeedForward(nn.Module):
    def __init__(self, seq, hidden_size, dropout:float=0.1):
        super().__init__()

        self.f1 = nn.Linear(seq, hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.f2 = nn.Linear(hidden_size, seq)
        
    def forward(self, x):
        x = self.f1(x)
        x = F.relu(x) # Considerar usar outra função de ativação
        x = self.dropout(x)
        x = self.f2(x)
        return x

In [114]:
class DecoderBlock(nn.Module):
    def __init__(self, max_seq_len, d_model, n_heads):
        super().__init__()

        # Defining Masked Multi Head Attetion
        self.masked_multi_head_attention = QKVAttention(
            d_model = d_model,
            h = n_heads,
            seq = max_seq_len,
            causal = True 
        )

        # Defining FeedForward (applies a layer normalization)
        self.masked_multi_head_attention = FeedForward(
            seq = max_seq_len,
            hidden_size = 4 * d_model
        )

        self.skip_connection_normalization = nn.LayerNorm(d_model)

    def forward(self, x):

        res = x 
        x = self.masked_multi_head_attention(x) # Masked Multi Head Attention
        x = self.skip_connection_normalization(x + res) # Layer Normalization
        
        res = x 
        x = self.masked_multi_head_attention(x) # Feed Forward
        x = self.skip_connection_normalization(x + res) # Residual + Layer Norm

        return x



class Transformer(nn.Module):
    def __init__(self, max_seq_len, vocab_size, embeding_dim, n_layers, d_model, n_heads):
        super().__init__()

        self.embeding = nn.Embedding(vocab_size, embeding_dim)
        
        self.pos_embeding = PosEmbeding(
            seq = max_seq_len,
            d_model = embeding_dim
        ).pe

        self.blocks = nn.ModuleList()
        for _ in range(n_layers):
            self.blocks.append(
                DecoderBlock(
                    d_model = d_model,
                    n_heads = n_heads,
                    max_seq_len = max_seq_len,
                    #causal=True # True if Masked
                )
            )

        self.linear = nn.Linear(d_model, d_model)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):

        input_embedding = self.embeding(x)

        positional_embedding = (input_embedding + self.pos_embeding)

        for transformer_block in self.blocks:
            x = transformer_block(positional_embedding)

        logits = self.linear(x)

        prob = self.softmax(logits)
    
        return prob



# Testando Código

In [115]:
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=d_model)
input_embedding = embedding_layer(input_ids)
positional_embedding = PosEmbeding(512,512).pe

x = input_embedding + positional_embedding

In [116]:
atencao = QKVAttention(512, 1, 512, True)
atencao(x)

tensor([[-0.1223,  0.4384,  0.1596,  ..., -0.2864, -0.5168, -0.2480],
        [-0.4732,  0.5691,  0.0532,  ..., -0.5666, -0.3851, -0.0705],
        [-0.6250,  0.3891, -0.0080,  ..., -0.6838, -0.4722,  0.0584],
        ...,
        [-0.0626,  0.3088, -0.1747,  ...,  0.1109,  0.0398,  0.2184],
        [-0.0611,  0.3072, -0.1693,  ...,  0.0994,  0.0404,  0.2181],
        [-0.0603,  0.3035, -0.1659,  ...,  0.0889,  0.0406,  0.2171]],
       grad_fn=<SqueezeBackward1>)

In [117]:
block = DecoderBlock(512, 512, 1)
block(x)

tensor([[[-1.1216,  0.0423,  1.9518,  ...,  0.2404, -2.1851, -2.3615],
         [ 0.0733,  0.7332,  1.1718,  ..., -1.8543,  0.5907, -1.5832],
         [-0.5964, -0.5975,  0.0208,  ...,  0.5518,  0.3661, -1.0394],
         ...,
         [ 0.4755,  0.4708,  0.3594,  ..., -0.3126,  0.0998,  1.0933],
         [ 0.9695,  0.2840,  0.5067,  ..., -0.4938, -0.0775,  1.2094],
         [ 0.8716, -0.6269, -0.1369,  ..., -0.2140,  0.1193,  1.2985]]],
       grad_fn=<NativeLayerNormBackward0>)

In [118]:
model = Transformer(512, vocab_size, 512, 6, 512, 8)
model

Transformer(
  (embeding): Embedding(30522, 512)
  (blocks): ModuleList(
    (0-5): 6 x DecoderBlock(
      (masked_multi_head_attention): FeedForward(
        (f1): Linear(in_features=512, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (f2): Linear(in_features=2048, out_features=512, bias=True)
      )
      (skip_connection_normalization): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    )
  )
  (linear): Linear(in_features=512, out_features=512, bias=True)
  (softmax): Softmax(dim=-1)
)

In [119]:
model(input_ids)

tensor([[[0.0036, 0.0014, 0.0016,  ..., 0.0038, 0.0021, 0.0006],
         [0.0078, 0.0015, 0.0065,  ..., 0.0011, 0.0012, 0.0017],
         [0.0016, 0.0024, 0.0023,  ..., 0.0058, 0.0020, 0.0016],
         ...,
         [0.0016, 0.0011, 0.0023,  ..., 0.0032, 0.0023, 0.0011],
         [0.0016, 0.0012, 0.0023,  ..., 0.0032, 0.0027, 0.0009],
         [0.0016, 0.0012, 0.0024,  ..., 0.0037, 0.0026, 0.0012]]],
       grad_fn=<SoftmaxBackward0>)